# Codec-Induced Domain Shift in Acoustic Event Classification
## Reproducibility Notebook - Band-Limiting Control and Degradation Decomposition

This notebook reproduces the band-limiting control experiment reported in the manuscript. It quantifies how much of the AMR-NB-induced performance drop is attributable to **bandwidth restriction** versus **codec-specific distortion**, using three conditions evaluated with an identical clean-trained classifier under ten-fold cross-validation:

| Condition | Signal processing | Isolates |
|-----------|-------------------|----------|
| A - Clean | None (original wideband audio) | Upper bound |
| B - Band-limited | 8 kHz resample, 3.4 kHz low-pass, 22.05 kHz | Bandwidth loss only |
| C - AMR-NB 4.75 kbit/s | Full encoder-decoder round trip | Bandwidth + codec artefacts |

The decomposition separates the two effects: (A - B) measures bandwidth loss, (B - C) measures codec-specific distortion.

### Reproducibility guarantees
- Fixed random seed (42) throughout.
- Condition A is recomputed here and cross-checked against the value reported in the main experiment; the run aborts if they diverge, guaranteeing consistency of claims.
- Every fold is checkpointed; interrupted sessions resume without recomputation.
- All datasets are mounted as Kaggle inputs; no network download is required at run time.

### Environment
Runtime: Kaggle, GPU T4. Datasets (added via **+ Add Input**): `gurjant-us8k-official`, `gurjant-esc50-official`, `gurjant-sc_v2-official`.

In [ ]:
# Configuration and reproducibility controls
import os, sys, json, random, subprocess, csv, warnings
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from IPython.display import display, HTML

warnings.filterwarnings("ignore", message="n_fft=.* is too large")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths (Kaggle)
IN      = Path("/kaggle/input")
WORK    = Path("/kaggle/working")
RESULTS = WORK / "results";   RESULTS.mkdir(parents=True, exist_ok=True)
FIGS    = RESULTS / "figs";   FIGS.mkdir(exist_ok=True)
CKPT    = RESULTS / "ckpt";   CKPT.mkdir(exist_ok=True)
PROC    = WORK / "processed"; PROC.mkdir(exist_ok=True)
TMP     = WORK / "tmp";       TMP.mkdir(exist_ok=True)

# Signal and training parameters (identical to the main experiment)
TARGET_SR = 22050
MAX_DUR   = 4.0
N_MELS    = 128
N_FFT     = 2048
HOP       = 512
N_FOLDS   = 10
BATCH     = 32
EPOCHS    = 30
PATIENCE  = 5
LR        = 1e-4
BANDS = [(0,500),(500,1000),(1000,2000),(2000,3400),
         (3400,4000),(4000,8000),(8000,11025)]

# Reference values from the main experiment (for the reproducibility cross-check)
REF_A_CLEAN_MEAN = 0.786   # ResNet-50 clean-trained, clean-tested, 10-fold mean
REF_C_AMR_MEAN   = 0.370   # ResNet-50 clean-trained, AMR-4.75k-tested, 10-fold mean
REF_TOLERANCE    = 0.030   # acceptable absolute deviation for a reproducibility match

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  CUDA devices: {torch.cuda.device_count()}")
assert DEVICE.type == "cuda", (
    "GPU not available. Enable it via Settings -> Accelerator -> GPU T4.")
print("Configuration loaded.")

In [ ]:
# Install AMR-NB-capable ffmpeg and Python dependencies
subprocess.run(
    "apt-get update -qq && apt-get install -y -qq "
    "libavcodec-extra libopencore-amrnb0 libopencore-amrwb0 ffmpeg",
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

encoders = subprocess.run(["ffmpeg","-encoders"],
                          capture_output=True, text=True).stdout
if "libopencore_amrnb" not in encoders:
    subprocess.run(
        "wget -q https://johnvansickle.com/ffmpeg/releases/"
        "ffmpeg-release-amd64-static.tar.xz && "
        "tar xf ffmpeg-release-amd64-static.tar.xz",
        shell=True)
    static = [d for d in os.listdir(".")
              if d.startswith("ffmpeg-") and d.endswith("-static")]
    if static:
        os.environ["PATH"] = os.path.abspath(static[0]) + ":" + os.environ["PATH"]
    encoders = subprocess.run(["ffmpeg","-encoders"],
                              capture_output=True, text=True).stdout

assert "libopencore_amrnb" in encoders, (
    "AMR-NB encoder unavailable. Ensure Internet is enabled in Settings.")

for pkg in ["librosa", "soundfile", "scikit-learn", "scipy", "statsmodels", "tqdm"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import librosa, soundfile as sf
print("AMR-NB encoder available. Dependencies installed.")

## 1. Dataset discovery
Datasets are located by searching the mounted Kaggle inputs. The UrbanSound8K loader handles both the official layout (`audio/foldN/`) and the flat layout (`foldN/`), so the notebook is agnostic to how the dataset was packaged.

In [ ]:
# Locate the three corpora among the mounted Kaggle inputs
def find_file(pattern):
    hits = list(IN.rglob(pattern))
    return hits[0] if hits else None

# UrbanSound8K (primary corpus for classification)
us8k_csv = find_file("UrbanSound8K.csv")
assert us8k_csv, ("UrbanSound8K.csv not found. Add gurjant-us8k-official via + Add Input.")
us8k_base = us8k_csv.parent
us8k_official = (us8k_base / "audio" / "fold1").exists()
US8K_ROWS = []
with open(us8k_csv) as fh:
    for r in csv.DictReader(fh):
        fold = r["fold"]
        sub = f"audio/fold{fold}" if us8k_official else f"fold{fold}"
        US8K_ROWS.append((str(us8k_base / sub / r["slice_file_name"]),
                          r["class"], int(r["fold"])))
CLASSES = sorted(set(c for _, c, _ in US8K_ROWS))
print(f"UrbanSound8K : {len(US8K_ROWS)} clips, {len(CLASSES)} classes "
      f"({'official' if us8k_official else 'flat'} layout)")

# ESC-50 (secondary corpus for the acoustic mismatch analysis)
esc_csv = find_file("esc50.csv")
ESC_ROWS = []
if esc_csv:
    with open(esc_csv) as fh:
        first = next(csv.DictReader(fh))["filename"]
    esc_audio = find_file(first)
    if esc_audio:
        esc_dir = esc_audio.parent
        with open(esc_csv) as fh:
            for r in csv.DictReader(fh):
                ESC_ROWS.append((str(esc_dir / r["filename"]),
                                 r["category"], int(r["fold"])))
print(f"ESC-50       : {len(ESC_ROWS)} clips")

# Speech Commands (speech reference for the mismatch analysis)
KEYWORDS = ["yes","no","up","down","left","right","on","off","stop","go"]
SC_ROWS = []; _gi = 0; sc_root = None
for kw in KEYWORDS:
    hit = find_file(kw)
    if hit and hit.is_dir() and list(hit.glob("*.wav")):
        sc_root = hit.parent; break
if sc_root:
    for kw in KEYWORDS:
        d = sc_root / kw
        if not d.exists(): continue
        for f in sorted(d.glob("*.wav"))[:200]:
            SC_ROWS.append((str(f), kw, 1 + (_gi % 5))); _gi += 1
print(f"Speech Cmds  : {len(SC_ROWS)} clips")

assert len(US8K_ROWS) == 8732, (
    f"Expected 8732 UrbanSound8K clips, found {len(US8K_ROWS)}. Check the dataset.")
print("\nAll datasets located.")

## 2. Signal-processing pipeline
Two degradations are defined. **Band-limiting** reproduces the bandwidth restriction of narrowband telephony without any codec: the signal is downsampled to 8 kHz, low-pass filtered at 3.4 kHz, and returned to the analysis rate. **AMR-NB** applies the real codec through a genuine encoder-decoder round trip. The band-limiting pipeline mirrors the AMR-NB input path exactly, so that the only difference between conditions B and C is the codec itself.

In [ ]:
import librosa, soundfile as sf
from scipy.signal import butter, sosfiltfilt

def bandlimit(y, sr=TARGET_SR, down_sr=8000, lp_hz=3400):
    """Bandwidth restriction without codec processing."""
    y8  = librosa.resample(y, orig_sr=sr, target_sr=down_sr)
    sos = butter(8, lp_hz / (down_sr / 2), btype="low", output="sos")
    y8f = sosfiltfilt(sos, y8).astype(np.float32)
    return librosa.resample(y8f, orig_sr=down_sr, target_sr=sr)

def amr_nb(inp, out, kbps=4.75):
    """AMR-NB encode/decode round trip via libopencore-amrnb."""
    stem = str(out) + ".amr"
    enc = subprocess.run(
        ["ffmpeg","-y","-i",str(inp),"-ac","1","-ar","8000",
         "-b:a",f"{kbps}k","-c:a","libopencore_amrnb","-f","amr",stem],
        capture_output=True, timeout=20)
    if enc.returncode != 0:
        raise RuntimeError(f"AMR-NB encode failed: {Path(inp).name}")
    dec = subprocess.run(
        ["ffmpeg","-y","-i",stem,"-ar",str(TARGET_SR),"-ac","1",str(out)],
        capture_output=True, timeout=20)
    if Path(stem).exists(): os.remove(stem)
    if dec.returncode != 0:
        raise RuntimeError(f"AMR-NB decode failed: {Path(inp).name}")

print("Signal-processing functions defined.")

## 3. Generate degraded corpora
Band-limited and AMR-NB versions of every clip are written once to the working directory and reused on subsequent runs. Processing is parallelised across CPU workers and is resumable: files already present are skipped, so an interrupted run continues from where it stopped.

In [ ]:
from tqdm.auto import tqdm
from multiprocessing.pool import ThreadPool
import multiprocessing as mp

def _process_clip(task):
    src, dst_bandlimit, dst_amr = task
    try:
        y, _ = librosa.load(src, sr=TARGET_SR, mono=True, duration=MAX_DUR)
    except Exception:
        return "error"
    if not Path(dst_bandlimit).exists():
        try: sf.write(dst_bandlimit, bandlimit(y), TARGET_SR)
        except Exception: return "error"
    if not Path(dst_amr).exists():
        tmp = TMP / f"_{Path(src).stem}.wav"
        try:
            sf.write(tmp, y, TARGET_SR); amr_nb(tmp, dst_amr)
        except Exception: return "error"
        finally:
            if tmp.exists(): tmp.unlink()
    return "ok"

(PROC / "bandlimit").mkdir(exist_ok=True)
(PROC / "amr475").mkdir(exist_ok=True)

tasks = []
for src, _, fold in US8K_ROWS:
    fb = PROC / f"bandlimit/fold{fold}"; fb.mkdir(exist_ok=True)
    fc = PROC / f"amr475/fold{fold}";   fc.mkdir(exist_ok=True)
    tasks.append((src, str(fb / Path(src).name), str(fc / Path(src).name)))

workers = max(1, mp.cpu_count() - 1)
print(f"Generating degraded corpora using {workers} workers ...")
with ThreadPool(workers) as pool:
    outcomes = list(tqdm(pool.imap(_process_clip, tasks, chunksize=32),
                         total=len(tasks), desc="Processing"))

n_bandlimit = len(list((PROC / "bandlimit").rglob("*.wav")))
n_amr       = len(list((PROC / "amr475").rglob("*.wav")))
print(f"Band-limited clips : {n_bandlimit}")
print(f"AMR-NB clips       : {n_amr}")
assert n_bandlimit == len(US8K_ROWS) and n_amr == len(US8K_ROWS), (
    "Some clips failed to process. Re-run this cell to complete them.")
print("Degraded corpora ready.")

## 4. Pipeline verification
Before training, the degradations are verified acoustically on a random sample. Two checks are reported: the per-band energy change of each condition relative to clean, and the high-band (>3.4 kHz) attenuation. The AMR-NB high-band energy fraction is asserted to be small, which confirms the codec was genuinely applied rather than passed through.

In [ ]:
import random as _rnd
_rnd.seed(SEED)

def band_energy(y, sr=TARGET_SR):
    S = np.abs(librosa.stft(y, n_fft=1024)) ** 2
    f = librosa.fft_frequencies(sr=sr, n_fft=1024)
    return {b: float(S[(f >= b[0]) & (f < b[1])].sum()) for b in BANDS}

def delta_db(degraded, clean):
    return {f"{b[0]}-{b[1]}": float(10 * np.log10((degraded[b] + 1e-12) /
                                                  (clean[b] + 1e-12))) for b in BANDS}

def hf_attenuation(clean, degraded, cut=3400, sr=TARGET_SR):
    def high(y):
        S = np.abs(librosa.stft(y, n_fft=1024)) ** 2
        f = librosa.fft_frequencies(sr=sr, n_fft=1024)
        return S[f >= cut].sum()
    n = min(len(clean), len(degraded))
    return float(10 * np.log10((high(degraded[:n]) + 1e-12) /
                               (high(clean[:n]) + 1e-12)))

sample = _rnd.sample(US8K_ROWS, 50)
bl_bands, amr_bands, bl_hf, amr_hf = [], [], [], []
for src, _, fold in sample:
    p_bl  = PROC / f"bandlimit/fold{fold}" / Path(src).name
    p_amr = PROC / f"amr475/fold{fold}" / Path(src).name
    if not (p_bl.exists() and p_amr.exists()): continue
    yc, _ = librosa.load(src, sr=TARGET_SR, duration=MAX_DUR)
    yb, _ = librosa.load(str(p_bl),  sr=TARGET_SR, duration=MAX_DUR)
    ya, _ = librosa.load(str(p_amr), sr=TARGET_SR, duration=MAX_DUR)
    ec = band_energy(yc)
    bl_bands.append(delta_db(band_energy(yb), ec))
    amr_bands.append(delta_db(band_energy(ya), ec))
    bl_hf.append(hf_attenuation(yc, yb))
    amr_hf.append(hf_attenuation(yc, ya))

bl_hf_mean, amr_hf_mean = float(np.mean(bl_hf)), float(np.mean(amr_hf))
print(f"{'Band (Hz)':>12} | {'Band-limited':>12} | {'AMR-NB':>10} | {'Codec-only':>10}")
print("-" * 54)
for b in BANDS:
    k = f"{b[0]}-{b[1]}"
    mb = np.mean([d[k] for d in bl_bands])
    ma = np.mean([d[k] for d in amr_bands])
    print(f"{k:>12} | {mb:>+12.1f} | {ma:>+10.1f} | {ma - mb:>+10.1f}")
print("-" * 54)
print(f"High-band (>3.4 kHz) attenuation:  band-limited {bl_hf_mean:+.1f} dB, "
      f"AMR-NB {amr_hf_mean:+.1f} dB")

# Confirm the codec genuinely removed high-frequency content
amr_files = list((PROC / "amr475").rglob("*.wav"))[:10]
hf_fraction = []
for f in amr_files:
    y, _ = librosa.load(str(f), sr=TARGET_SR)
    S = np.abs(librosa.stft(y)) ** 2
    freq = librosa.fft_frequencies(sr=TARGET_SR)
    hf_fraction.append(S[freq >= 3400].sum() / (S.sum() + 1e-12))
assert np.mean(hf_fraction) < 0.01, (
    f"AMR-NB high-band fraction {np.mean(hf_fraction):.4f} is too high; "
    "the codec may not have been applied.")
print(f"AMR-NB high-band energy fraction: {np.mean(hf_fraction):.4f}  (codec confirmed)")

json.dump({"bandlimit_hf_db": bl_hf_mean, "amr_hf_db": amr_hf_mean,
           "codec_specific_db": amr_hf_mean - bl_hf_mean,
           "per_band_bandlimit": {f"{b[0]}-{b[1]}": float(np.mean([d[f"{b[0]}-{b[1]}"] for d in bl_bands])) for b in BANDS},
           "per_band_amr": {f"{b[0]}-{b[1]}": float(np.mean([d[f"{b[0]}-{b[1]}"] for d in amr_bands])) for b in BANDS}},
          open(RESULTS / "acoustic_verification.json", "w"), indent=2)
print("Acoustic verification saved.")

## 5. Model and cross-validation utilities
The classifier is an ImageNet-pretrained ResNet-50 adapted to single-channel mel-spectrogram input, matching the architecture used in the main experiment. Training uses early stopping on a held-out validation split drawn from the training folds; the test fold is never seen during training or model selection. Each fold result is written to a checkpoint file and reloaded if present.

In [ ]:
def to_mel(path):
    y, _ = librosa.load(path, sr=TARGET_SR, mono=True, duration=MAX_DUR)
    length = int(TARGET_SR * MAX_DUR)
    y = np.pad(y, (0, max(0, length - len(y))))[:length]
    mel = librosa.power_to_db(
        librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_mels=N_MELS,
                                       n_fft=N_FFT, hop_length=HOP),
        ref=np.max)
    return ((mel - mel.mean()) / (mel.std() + 1e-6)).astype(np.float32)

class MelDataset(Dataset):
    def __init__(self, files, labels):
        self.files, self.labels = files, labels
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        return torch.tensor(to_mel(self.files[i])).unsqueeze(0), self.labels[i]

def build_split(rows, condition_dir, folds, classes):
    label_index = {c: i for i, c in enumerate(classes)}
    files, labels = [], []
    for src, cls, fold in rows:
        if fold not in folds: continue
        path = src if condition_dir is None else \
               str(Path(condition_dir) / f"fold{fold}" / Path(src).name)
        if not Path(path).exists(): continue
        files.append(path); labels.append(label_index[cls])
    if not files:
        raise RuntimeError(f"No clips for folds={folds}, condition={condition_dir}")
    return MelDataset(files, labels)

class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.to_rgb = nn.Conv2d(1, 3, kernel_size=1, bias=False)
        backbone = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
        for p in backbone.parameters(): p.requires_grad = False
        for p in backbone.layer4.parameters(): p.requires_grad = True
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.embedding = nn.Sequential(nn.Flatten(), nn.Linear(2048, 128),
                                       nn.ReLU(), nn.Dropout(0.3))
        self.classifier = nn.Linear(128, num_classes)
    def forward(self, x):
        return self.classifier(self.embedding(self.features(self.to_rgb(x))))

def train_and_evaluate_fold(train_ds, test_ds, num_classes, seed, tag):
    checkpoint = CKPT / f"{tag}.json"
    if checkpoint.exists():
        result = json.load(open(checkpoint))
        print(f"  {tag}: loaded from checkpoint (F1 = {result['macro_f1']:.3f})")
        return result
    torch.manual_seed(seed)
    model = ResNet50Classifier(num_classes).to(DEVICE)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    criterion = nn.CrossEntropyLoss()
    n_val = max(1, len(train_ds) // 10)
    n_train = len(train_ds) - n_val
    train_subset, val_subset = torch.utils.data.random_split(
        train_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(seed))
    train_loader = DataLoader(train_subset, batch_size=BATCH, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH, shuffle=False,
                            num_workers=2, pin_memory=True)
    best_val, patience_counter, best_state = float("inf"), 0, None
    for _ in range(EPOCHS):
        model.train()
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(X), y).backward()
            optimizer.step()
        model.eval(); val_loss = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                val_loss += criterion(model(X.to(DEVICE)), y.to(DEVICE)).item()
        val_loss /= len(val_loader)
        if val_loss < best_val:
            best_val, patience_counter = val_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE: break
    model.load_state_dict(best_state); model.eval()
    preds, truths = [], []
    test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False,
                             num_workers=2, pin_memory=True)
    with torch.no_grad():
        for X, y in test_loader:
            preds += model(X.to(DEVICE)).argmax(1).cpu().tolist()
            truths += y.tolist()
    result = {"macro_f1": float(f1_score(truths, preds, average="macro", zero_division=0)),
              "per_class": f1_score(truths, preds, average=None, zero_division=0).tolist()}
    json.dump(result, open(checkpoint, "w"))
    del model, best_state; torch.cuda.empty_cache()
    print(f"  {tag}: F1 = {result['macro_f1']:.3f}")
    return result

print("Model and cross-validation utilities defined.")

## 6. Condition A - reproducibility cross-check
Condition A (clean training, clean testing) is recomputed here under ten-fold cross-validation and compared against the value reported in the main experiment. Agreement within tolerance confirms that this notebook reproduces the main pipeline and that the decomposition below rests on a consistent baseline. The run aborts on divergence.

In [ ]:
folds = list(range(1, N_FOLDS + 1))
num_classes = len(CLASSES)
results = {}

print("Condition A - clean training, clean testing")
a_scores = []
for test_fold in folds:
    train_folds = [f for f in folds if f != test_fold]
    train_ds = build_split(US8K_ROWS, None, train_folds, CLASSES)
    test_ds  = build_split(US8K_ROWS, None, [test_fold], CLASSES)
    r = train_and_evaluate_fold(train_ds, test_ds, num_classes, SEED,
                                f"A_clean_f{test_fold}")
    a_scores.append(r["macro_f1"])
results["A_clean"] = {"mean": float(np.mean(a_scores)),
                      "std": float(np.std(a_scores)),
                      "f1all": a_scores}

deviation = abs(results["A_clean"]["mean"] - REF_A_CLEAN_MEAN)
print(f"\nCondition A mean F1 : {results['A_clean']['mean']:.3f} "
      f"(main experiment: {REF_A_CLEAN_MEAN:.3f}, deviation {deviation:.3f})")
assert deviation <= REF_TOLERANCE, (
    f"Condition A deviates from the main experiment by {deviation:.3f}, "
    f"exceeding the {REF_TOLERANCE:.3f} tolerance. Investigate before proceeding.")
print("Reproducibility cross-check passed.")

## 7. Conditions B and C - the band-limiting decomposition
Both conditions use the identical clean-trained classifier evaluated on degraded test folds: condition B on band-limited audio, condition C on AMR-NB audio. Comparing them against condition A isolates the two components of the degradation.

In [ ]:
condition_dirs = {"B_bandlimit": str(PROC / "bandlimit"),
                  "C_amr475":    str(PROC / "amr475")}

for name, condition_dir in condition_dirs.items():
    label = "band-limited" if name.startswith("B") else "AMR-NB 4.75 kbit/s"
    print(f"Condition {name[0]} - clean training, {label} testing")
    scores = []
    for test_fold in folds:
        train_folds = [f for f in folds if f != test_fold]
        train_ds = build_split(US8K_ROWS, None, train_folds, CLASSES)
        test_ds  = build_split(US8K_ROWS, condition_dir, [test_fold], CLASSES)
        r = train_and_evaluate_fold(train_ds, test_ds, num_classes, SEED,
                                    f"{name}_f{test_fold}")
        scores.append(r["macro_f1"])
    results[name] = {"mean": float(np.mean(scores)),
                     "std": float(np.std(scores)),
                     "f1all": scores}
    print()

# Cross-check condition C against the main experiment
c_deviation = abs(results["C_amr475"]["mean"] - REF_C_AMR_MEAN)
print(f"Condition C mean F1 : {results['C_amr475']['mean']:.3f} "
      f"(main experiment: {REF_C_AMR_MEAN:.3f}, deviation {c_deviation:.3f})")
assert c_deviation <= REF_TOLERANCE, (
    f"Condition C deviates from the main experiment by {c_deviation:.3f}, "
    f"exceeding the {REF_TOLERANCE:.3f} tolerance.")
print("Condition C consistency check passed.")

json.dump(results, open(RESULTS / "bandlimit_comparison.json", "w"), indent=2)
print("\nCondition summary")
for name in ["A_clean", "B_bandlimit", "C_amr475"]:
    print(f"  {name:12} F1 = {results[name]['mean']:.3f} +/- {results[name]['std']:.3f}")

## 8. Statistical analysis
The decomposition is quantified with paired tests over the ten fold-level scores. Bandwidth loss (A - B) and codec-specific distortion (B - C) are reported as absolute F1 differences with bootstrap 95% confidence intervals and Holm-corrected Wilcoxon p-values. The share of the total drop attributable to each component is reported as a percentage.

In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

A = np.array(results["A_clean"]["f1all"])
B = np.array(results["B_bandlimit"]["f1all"])
C = np.array(results["C_amr475"]["f1all"])

def bootstrap_ci(values, n_boot=5000):
    draws = [np.mean(np.random.choice(values, len(values), replace=True))
             for _ in range(n_boot)]
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

rng = np.random.default_rng(SEED)
np.random.seed(SEED)
p_values = [wilcoxon(A, B).pvalue,
            wilcoxon(A, C).pvalue,
            wilcoxon(B, C).pvalue if not np.allclose(B, C) else 1.0]
_, p_holm, _, _ = multipletests(p_values, method="holm")

total_drop      = float(A.mean() - C.mean())
bandwidth_drop  = float(A.mean() - B.mean())
codec_drop      = float(B.mean() - C.mean())
bandwidth_share = 100 * bandwidth_drop / (total_drop + 1e-12)
codec_share     = 100 * codec_drop / (total_drop + 1e-12)

acoustic = json.load(open(RESULTS / "acoustic_verification.json"))
statistics = {
    "A_vs_B": {"mean_diff": float((A - B).mean()), "ci": bootstrap_ci(A - B),
               "p_holm": float(p_holm[0])},
    "A_vs_C": {"mean_diff": float((A - C).mean()), "ci": bootstrap_ci(A - C),
               "p_holm": float(p_holm[1])},
    "B_vs_C": {"mean_diff": float((B - C).mean()), "ci": bootstrap_ci(B - C),
               "p_holm": float(p_holm[2])},
    "decomposition": {"total_drop": total_drop, "bandwidth_drop": bandwidth_drop,
                      "codec_drop": codec_drop, "bandwidth_pct": bandwidth_share,
                      "codec_pct": codec_share}}
json.dump(statistics, open(RESULTS / "bandlimit_statistics.json", "w"), indent=2)

print("Degradation decomposition (ResNet-50, ten-fold cross-validation)")
print("-" * 60)
print(f"  Clean (A)          : {A.mean():.3f} +/- {A.std():.3f}")
print(f"  Band-limited (B)   : {B.mean():.3f} +/- {B.std():.3f}")
print(f"  AMR-NB 4.75k (C)   : {C.mean():.3f} +/- {C.std():.3f}")
print("-" * 60)
print(f"  Total drop (A-C)         : {total_drop:.3f} F1")
print(f"  Bandwidth loss (A-B)     : {bandwidth_drop:.3f} F1  ({bandwidth_share:.1f}%)")
print(f"  Codec distortion (B-C)   : {codec_drop:.3f} F1  ({codec_share:.1f}%)")
print("-" * 60)
sig = "significant" if statistics["B_vs_C"]["p_holm"] < 0.05 else "not significant at n=10"
ci = statistics["B_vs_C"]["ci"]
print(f"  Codec effect (B vs C): diff {statistics['B_vs_C']['mean_diff']:+.3f}, "
      f"95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}], Holm p = {statistics['B_vs_C']['p_holm']:.4f} ({sig})")
print(f"  Codec-specific high-band attenuation: {acoustic['codec_specific_db']:+.1f} dB")

## 9. Figures
Two figures are produced: the three-condition comparison with the decomposition annotated, and the per-band energy profile distinguishing bandwidth loss from codec-specific distortion. Both are written to the results directory in PDF and PNG.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "serif", "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.3,
                     "savefig.bbox": "tight"})

stats = json.load(open(RESULTS / "bandlimit_statistics.json"))
decomp = stats["decomposition"]

# Figure 1: three-condition comparison
fig, ax = plt.subplots(figsize=(5.5, 4))
labels = ["A: Clean", "B: Band-limited\n(bandwidth only)",
          "C: AMR-NB 4.75k\n(bandwidth + codec)"]
positions = np.array([0, 1, 2])
colours = ["#228833", "#4477AA", "#EE6677"]
means = [A.mean(), B.mean(), C.mean()]
stds  = [A.std(), B.std(), C.std()]
ax.bar(positions, means, color=colours, alpha=0.85, width=0.55)
ax.errorbar(positions, means, yerr=stds, fmt="none", color="black", capsize=5)
for pos, values in zip(positions, [A, B, C]):
    jitter = np.random.uniform(-0.1, 0.1, len(values))
    ax.scatter(pos + jitter, values, s=16, color="black", alpha=0.4, zorder=5)
ax.annotate("", xy=(1, B.mean()), xytext=(0, A.mean()),
            arrowprops=dict(arrowstyle="->", color="#4477AA", lw=1.5))
ax.annotate("", xy=(2, C.mean()), xytext=(1, B.mean()),
            arrowprops=dict(arrowstyle="->", color="#EE6677", lw=1.5))
ax.text(0.5, (A.mean() + B.mean()) / 2,
        f"-{decomp['bandwidth_drop']:.3f}\n({decomp['bandwidth_pct']:.0f}%)",
        ha="center", va="center", fontsize=8, color="#4477AA")
ax.text(1.5, (B.mean() + C.mean()) / 2,
        f"-{decomp['codec_drop']:.3f}\n({decomp['codec_pct']:.0f}%)",
        ha="center", va="center", fontsize=8, color="#EE6677")
ax.set_xticks(positions); ax.set_xticklabels(labels, fontsize=8.5)
ax.set_ylabel("Macro-F1"); ax.set_ylim(0.2, 0.95)
ax.set_title("Bandwidth restriction versus codec-specific distortion")
plt.tight_layout()
plt.savefig(FIGS / "fig_bandlimit_comparison.pdf")
plt.savefig(FIGS / "fig_bandlimit_comparison.png", dpi=150)
plt.close()

# Figure 2: per-band energy profile
acoustic = json.load(open(RESULTS / "acoustic_verification.json"))
bl_profile  = [acoustic["per_band_bandlimit"][f"{b[0]}-{b[1]}"] for b in BANDS]
amr_profile = [acoustic["per_band_amr"][f"{b[0]}-{b[1]}"] for b in BANDS]
band_labels = [f"{b[0]}-{b[1]}" for b in BANDS]
fig, ax = plt.subplots(figsize=(6.5, 3.4))
x = np.arange(len(BANDS))
ax.plot(x, bl_profile,  marker="s", color="#4477AA", lw=1.5, label="Band-limited")
ax.plot(x, amr_profile, marker="o", color="#EE6677", lw=1.5, label="AMR-NB 4.75k")
ax.fill_between(x, bl_profile, amr_profile, alpha=0.15, color="#EE6677",
                label="Codec-specific")
ax.axhline(0, color="black", lw=0.6)
ax.axvline(3.5, ls=":", color="grey", lw=1)
ax.text(3.6, min(amr_profile) - 4, "Nyquist\nlimit", fontsize=7, color="grey")
ax.set_xticks(x); ax.set_xticklabels(band_labels, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Energy change (dB)")
ax.set_title("Per-band energy change relative to clean audio")
ax.legend(fontsize=7.5)
plt.tight_layout()
plt.savefig(FIGS / "fig_bandlimit_acoustic.pdf")
plt.savefig(FIGS / "fig_bandlimit_acoustic.png", dpi=150)
plt.close()
print("Figures written to", FIGS)

## 10. Results summary and archive
A machine-readable summary and a text block suitable for the manuscript are produced, and all outputs are archived into a single downloadable file.

In [ ]:
import shutil

summary = json.load(open(RESULTS / "bandlimit_comparison.json"))
stats   = json.load(open(RESULTS / "bandlimit_statistics.json"))
acoustic = json.load(open(RESULTS / "acoustic_verification.json"))
decomp  = stats["decomposition"]
codec_significant = stats["B_vs_C"]["p_holm"] < 0.05

manifest = {
    "conditions": {k: {"mean": summary[k]["mean"], "std": summary[k]["std"]}
                   for k in ["A_clean", "B_bandlimit", "C_amr475"]},
    "decomposition": decomp,
    "codec_effect_significant": bool(codec_significant),
    "acoustic_codec_specific_db": acoustic["codec_specific_db"],
    "seed": SEED, "n_folds": N_FOLDS}
json.dump(manifest, open(RESULTS / "manifest.json", "w"), indent=2)

clause = ("and is statistically significant "
          f"(Holm-corrected Wilcoxon p = {stats['B_vs_C']['p_holm']:.3f})"
          if codec_significant else
          "though it does not reach significance at ten folds "
          f"(Holm-corrected Wilcoxon p = {stats['B_vs_C']['p_holm']:.3f})")

paragraph = (
    "To separate bandwidth restriction from codec-specific distortion, we evaluated a "
    "band-limited control (condition B) in which each clip was resampled to 8 kHz, "
    "low-pass filtered at 3.4 kHz (eighth-order Butterworth), and resampled back to "
    "22.05 kHz, reproducing the bandwidth of AMR-NB without codec-specific processing. "
    f"The clean-trained ResNet-50 attains macro-F1 of {summary['A_clean']['mean']:.3f} on "
    f"clean audio, {summary['B_bandlimit']['mean']:.3f} on band-limited audio, and "
    f"{summary['C_amr475']['mean']:.3f} on AMR-NB audio at 4.75 kbit/s. Of the total "
    f"{decomp['total_drop']:.3f}-point degradation, {decomp['bandwidth_pct']:.0f}% "
    f"({decomp['bandwidth_drop']:.3f} points) is attributable to bandwidth restriction and "
    f"{decomp['codec_pct']:.0f}% ({decomp['codec_drop']:.3f} points) to codec-specific "
    f"processing, {clause}. Acoustically, band-limiting attenuates energy above 3.4 kHz by "
    f"{acoustic['bandlimit_hf_db']:+.1f} dB, while AMR-NB adds a further "
    f"{acoustic['codec_specific_db']:+.1f} dB of codec-specific attenuation.")

(RESULTS / "manuscript_paragraph.txt").write_text(paragraph)

archive = shutil.make_archive(str(WORK / "bandlimit_results"), "zip", str(RESULTS))
print("Manuscript paragraph:\n")
print(paragraph)
print("\nArchive written:", archive)
print("Contents:", sorted(p.name for p in RESULTS.glob("*")))